# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-12_base_vs_adapter.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker — see the [course README](https://github.com/justkrismanohar/fine-tuning-build/blob/main/README.md).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day3/T1-12_base_vs_adapter.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day3"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day3" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


Full live scoreboards need a GPU + baked/prebuilt adapter. Without them the notebook loads RECORDED files and labels them.


# T1-12 Full held-out scoring: base vs floor vs adapter

**Time:** 10:45–12:00 &nbsp;|&nbsp; **Evidence:** criteria scoreboard + redaction HOLD / PASS CHECKS

Nothing here promises the adapter is better. Read what was measured.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** change only lines marked `# [CHANGE]`. Allowed values are named in the cell above.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path.

Git is not required. If you are stuck, load `completed/` for **this** notebook only and mark it as loaded from reference.


**Data ban:** CivicDesk fiction only. `heldout/cases.py` is a **test** set — do not train on it.


## 0. Backend


In [ ]:
# Colab track: demo/RECORDED. Free T4 may still run TrainSession if CUDA is free.
BACKEND = "demo"  # "olmo" | "phi" | "demo"

import civicdesk_ft as ft

cfg = ft.set_backend(BACKEND)


## 1. Adapter card from a real training_report.json

What this cell does: load the adapter T1-11 saved (or prebuilt) and print its measured card — not a stub.


In [ ]:
# [CHANGE] same value as T1-11 section 7, or leave "" to read logs/last_adapter.json
ADAPTER = ""  # e.g. "adapters/my_criteria_run" or "prebuilt"

import civicdesk_ft as ft
import ft_train
from pathlib import Path
import json

if not ADAPTER:
    last = Path("logs/last_adapter.json")
    if last.is_file():
        ADAPTER = json.loads(last.read_text(encoding="utf-8"))["adapter"]
        print("from last_adapter.json:", ADAPTER)
    else:
        ADAPTER = "prebuilt"
        print("no last_adapter.json — trying prebuilt")

try:
    model = ft_train.load_adapter(ADAPTER, task="criteria")
    ft.print_adapter_card(model["path"])
except Exception as exc:
    model = None
    print("Adapter load failed:", exc)


## 2. Criteria — all 18 held-out items, live

Columns: valid JSON rate; label agreement with gold; silence wrongly marked `does_not_meet`; `meets` with evidence not in the résumé.

Rows: base; base + evidence floor; adapter.

18 items from 7 résumés is a **small** sample — direction, not proof.


In [ ]:
# [RUN]
import ft_train

try:
    if model is None:
        raise RuntimeError("no model")
    compare = ft_train.compare_live(model, "criteria")
    board = ft_train.criteria_scoreboard(compare)
except Exception as exc:
    print("Live full compare unavailable:", exc)
    try:
        compare = ft_train.load_recorded("criteria")
        board = ft_train.criteria_scoreboard(compare)
    except Exception as exc2:
        print(exc2)
        compare = None
        board = None


## 3. Simplest thing first

**Ask:** does the evidence floor alone close most of the gap? If yes, say so — that is an honest result.


In [ ]:
# [RUN]
# Floor is already a row in the scoreboard above. Optional: re-print from pre-flight base only.
import civicdesk_ft as ft

ft.floor_from_preflight()


## 4. Redaction — same 40 documents as Day 1

Per document: **HOLD** or **PASS CHECKS**. Per run: **meets** or **does not meet** the example rule.

Live both ways uses the prebuilt redaction adapter in-process. Give it time on a T4, or set `DOC_LIMIT` for a subset.


In [ ]:
# [CHANGE] None = all 40 (slow). Or an int for a quick subset of document ids from the benchmark.
DOC_LIMIT = 3  # set None for the full bake compare

import civicdesk_ft as ft
from pathlib import Path

try:
    if DOC_LIMIT is None:
        cmp = ft.compare_runs(ft.baseline_run_path(), ft.adapter_run_path())
        print("RECORDED / measured file compare (olmo_adapter_v1.json) if that file exists.")
    else:
        import redaction_eval as R
        import sys
        sys.path.insert(0, str(Path("../shared/resume_redaction").resolve()))
        bench = R.load_benchmark()
        ids = [d["id"] for d in bench["docs"][: int(DOC_LIMIT)]]
        print("subset", ids)
        redact = ft.load_redactor("prebuilt")
        for did in ids:
            doc = next(d for d in bench["docs"] if d["id"] == did)
            b = redact(doc["text"], use_adapter=False)
            a = redact(doc["text"], use_adapter=True)
            print("=" * 40, did)
            print("base:", b[:200])
            print("adapter:", a[:200])
            print("score base", R.score_doc(doc, b))
            print("score adapter", R.score_doc(doc, a))
        cmp = {"note": "subset live", "docs": ids}
    ft.save_log("T1-12_compare", cmp if isinstance(cmp, dict) else {"ok": True})
except Exception as exc:
    print("Redaction compare unavailable:", exc)
    cmp = None


## 5. Two documents both ways

What this cell does: `DOC_IDS` from the Day 1 40-document file, base vs adapter via the in-process toggle.


In [ ]:
# [CHANGE] test ids from resume_redaction_test_v1.json
DOC_IDS = ["B01", "B12"]

import civicdesk_ft as ft

try:
    redact = ft.load_redactor("prebuilt")
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path("../shared/resume_redaction").resolve()))
    import redaction_eval as R
    bench = R.load_benchmark()
    by = {d["id"]: d for d in bench["docs"]}
    for did in DOC_IDS:
        doc = by[did]
        print("=" * 60, did)
        for flag, tag in ((False, "base"), (True, "adapter")):
            out = redact(doc["text"], use_adapter=flag)
            sc = R.score_doc(doc, out)
            print("---", tag, "---")
            print(out[:400])
            print("leaked", sc["leaked"], "over_redacted", sc["over_redacted"])
except Exception as exc:
    print("live unavailable:", exc)


## 6. Failure analysis

List three cases the **adapter** still gets wrong. Cause guess: data gap, label rule, or layout.


In [ ]:
# [CHANGE] after you read the printout
STILL_WRONG = [
    # {"id": "R2-C3", "cause": "label rule"},
]

import civicdesk_ft as ft

ft.still_wrong_criteria(compare if "compare" in dir() else None, which="adapter")
print("your list:", STILL_WRONG)


## 7. Probes — an adapter is not a database

What this cell does: ask a handbook question with **no** sources, through base and adapter. Then unfamiliar layouts. Tickets belong in T1-13.


In [ ]:
# [RUN]
import json
from pathlib import Path
import ft_train

probe = json.loads(Path("data/demo/not_a_database_probe.json").read_text(encoding="utf-8"))
q = probe["question"]
print("question:", q)
if model is not None:
    # custom one-off: reuse criteria system is wrong; just generate both ways with a short system
    system = "You answer CivicDesk desk questions. If you have no source, say you do not know. Do not invent a circular."
    with model["model"].disable_adapter():
        base = ft_train._generate(model, system, q, 120)
    ad = ft_train._generate(model, system, q, 120)
    print("LIVE, this session")
    print("--- base ---"); print(base)
    print("--- adapter ---"); print(ad)
else:
    print("RECORDED / demo text from probe file (no live model):")
    print("base:", probe.get("base_demo"))
    print("adapter:", probe.get("adapter_demo"))
print("An adapter does not look up a handbook. That was Day 2.")


In [ ]:
# [OPTIONAL] unfamiliar layouts through the criteria adapter
import json
from pathlib import Path

docs = json.loads(Path("data/demo/unfamiliar_layouts.json").read_text(encoding="utf-8"))
print(docs.get("note"))
for d in docs["docs"]:
    print("---", d["id"], d["layout"], "---")
    print(d["text"][:300])


In [ ]:
# [OPTIONAL] regex floor on one stress CV
import civicdesk_ft as ft

ft.print_redaction_floor("R2")


## Ending — one-line verdict

**Handback template:** For criteria, the [prompt / rule floor / adapter] is the simplest thing that met [metric]. A human still decides because [reason].


In [ ]:
# [CHANGE] fill the brackets in your own words
VERDICT = (
    "For criteria, the [prompt / rule floor / adapter] is the simplest thing that met [metric]. "
    "A human still decides because [reason]."
)
print(VERDICT)
import civicdesk_ft as ft

ft.save_log("T1-12_verdict", {"verdict": VERDICT, "adapter": ADAPTER})
